# 13. Alternatives we tried, under the paper's exact protocol (CPU only)

**Question:** would a standard fix on the similarity table work as well as OAF?

We test three label-free alternatives, each on its own and combined with OAF:
- **camera z-norm**: scores normalised per camera pair;
- **camera S-norm**: scores normalised per clip and camera;
- **k-reciprocal re-ranking** (Zhong et al., CVPR 2017).

**Same rules as the main results (Table 4):**
- the threshold is set on one half of the people (largest cut with pair precision ≥ 90%) and tested on the other half;
- 25 splits (seed 2027) × 2 directions = 50 tests.

Normalisation and re-ranking use only the clips of each half, and no labels.

**Needs (already on Drive):** your CSCI all-frames and OAF fingerprints, and the ThreeModels fingerprints (FastReID, OSNet).

**Runtime:** about 5–10 min on a CPU runtime. Each model is saved to Drive as it finishes, so a restart resumes.

**Output:** `MEVID-results/Alternatives-SameClothes`.

**Check before trusting the numbers:** the "all frames" and "OAF" rows must match Table 4 of the paper (CSCI 81.66 → 85.77, FastReID 41.72 → 46.68, OSNet 40.97 → 48.73).


**Paper:** Sec. 6, paragraph "Alternatives we tried"; supplementary Tables S1 and S2.

In [ ]:
# 1. Setup: Drive and your real fingerprints (MEVID-SC test, 1,368 clips)
import os, glob, json, numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from google.colab import drive
from tqdm.auto import tqdm
drive.mount('/content/drive')
SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(pats[0])
f_all = find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/test_templates_csci_sc.npz',
              f'{SD}/**/CSCI-SameClothes/test_templates_csci_sc.npz'])
f_oaf = find([f'{SD}/Data/MEVID-results/OcclusionAware-SameClothes/test_templates_csci_visible.npz',
              f'{SD}/**/OcclusionAware-SameClothes/test_templates_csci_visible.npz'])
f_tm = {m: find([f'{SD}/Data/MEVID-results/ThreeModels-SameClothes/fingerprints_{m}.npz',
                 f'{SD}/**/ThreeModels-SameClothes/fingerprints_{m}.npz']) for m in ['FastReID', 'OSNet']}
OUT = os.path.join(os.path.dirname(os.path.dirname(f_all)), 'Alternatives-SameClothes')
os.makedirs(OUT, exist_ok=True)

a, o = np.load(f_all), np.load(f_oaf)
assert a['templates'].shape == o['templates'].shape == (1368, 1024), 'not the real MEVID-SC CSCI files'
assert (a['pid'] == o['pid']).all() and (a['cam'] == o['cam']).all()
DATA = {'CSCI': dict(all=a['templates'], oaf=o['templates'], pid=a['pid'], cam=a['cam'])}
for m, f in f_tm.items():
    z = np.load(f)
    assert (z['pid'] == a['pid']).all(), f'{m}: clip order differs from CSCI'
    DATA[m] = dict(all=z['all frames'], oaf=z['ours (pose: person visible)'], pid=z['pid'], cam=z['cam'])
SETTINGS = dict(n_splits=25, seed=2027, target_pair_precision=90, rerank=dict(k1=20, k2=6, lam=0.3))
json.dump(SETTINGS, open(f'{OUT}/SETTINGS_frozen.json', 'w'), indent=2)
for m, d in DATA.items():
    print(f"{m}: {d['all'].shape[0]} clips, {len(np.unique(d['pid']))} people, {len(np.unique(d['cam']))} cameras")
print('results ->', OUT)


In [ ]:
# 2. The protocol (identical to the paper) and the three alternatives
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment

CUTS = np.geomspace(0.001, 1.5, 120)


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)


def score_identities(pred, p, c):
    """Clip-level IDF1, IDs per person, pair precision, cross-camera recall."""
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1)); np.add.at(M, (pi, ci), 1)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0); tp, same = (M * (M - 1)).sum(), (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    Kc = np.zeros((M.shape[0], ki.max() + 1, M.shape[1])); np.add.at(Kc, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (Kc.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (Kc ** 2).sum((1, 2))).sum()
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'pair precision %': 100 * tp / same if same else 100.0,
            'cross-camera recall %': 100 * x_same / max(1, x_tot)}



def to_dist(S):           # any similarity -> distance in [0, 2], order kept
    lo, hi = S.min(), S.max(); D = 2 * (hi - S) / max(hi - lo, 1e-9)
    D = (D + D.T) / 2; np.fill_diagonal(D, 0); return D

def zpair(S, c):
    cams = np.unique(c); ci = np.searchsorted(cams, c); off = ~np.eye(len(S), dtype=bool)
    Z = np.zeros_like(S); g = S[off]
    for a in range(len(cams)):
        for b in range(a, len(cams)):
            m = (np.outer(ci == a, ci == b) | np.outer(ci == b, ci == a))
            v = S[m & off]; mu, sd = (v.mean(), v.std()) if len(v) > 10 else (g.mean(), g.std())
            Z[m] = (S[m] - mu) / max(sd, 1e-6)
    return Z

def snorm(S, c):
    cams = np.unique(c); ci = np.searchsorted(cams, c); S2 = S.astype(np.float64).copy(); np.fill_diagonal(S2, np.nan)
    gmu, gsd = np.nanmean(S2), np.nanstd(S2); mu = np.full((len(S), len(cams)), gmu); sd = np.full_like(mu, gsd)
    for k in range(len(cams)):
        blk = S2[:, ci == k]; cnt = (~np.isnan(blk)).sum(1); ok = cnt >= 5
        mu[ok, k] = np.nanmean(blk[ok], 1); sd[ok, k] = np.maximum(np.nanstd(blk[ok], 1), 1e-6)
    a = (S - mu[:, ci]) / sd[:, ci]
    return (a + a.T) / 2

def rerank(S, k1=20, k2=6, lam=0.3):   # Zhong et al., CVPR 2017
    n = len(S); D = np.clip(2 - 2 * S, 0, None); D = D / D.max(); rank = np.argsort(D, 1)
    V = np.zeros((n, n), np.float32); h = int(round(k1 / 2))
    for i in range(n):
        fwd = rank[i, :k1 + 1]; kr = fwd[np.where((rank[fwd, :k1 + 1] == i).any(1))[0]]; ex = kr
        for cnd in kr:
            cf = rank[cnd, :h + 1]; ck = cf[np.where((rank[cf, :h + 1] == cnd).any(1))[0]]
            if len(np.intersect1d(ck, kr)) > 2 / 3 * len(ck): ex = np.append(ex, ck)
        ex = np.unique(ex); w = np.exp(-D[i, ex]); V[i, ex] = w / w.sum()
    if k2 > 1: V = np.stack([V[rank[i, :k2]].mean(0) for i in range(n)])
    J = np.zeros((n, n), np.float32)
    for i in range(n):
        nz = np.flatnonzero(V[i]); m = np.minimum(V[i, nz][None, :], V[:, nz]).sum(1); J[i] = 1 - m / (2 - m)
    J = (J + J.T) / 2
    return -((1 - lam) * J + lam * D)    # as a similarity

def plain(S, c): return S

def plain(S, c):
    return S

RR = lambda S, c: rerank(S, **SETTINGS['rerank'])
VARIANTS = {'all frames': ('all', plain), 'camera z-norm': ('all', zpair), 'camera S-norm': ('all', snorm),
            're-ranking': ('all', RR),
            'OAF': ('oaf', plain), 'OAF + camera z-norm': ('oaf', zpair), 'OAF + camera S-norm': ('oaf', snorm),
            'OAF + re-ranking': ('oaf', RR)}

def tree(D):
    return linkage(squareform(D, checks=False), 'average')

def calib(L, p, c):
    best = CUTS[0]
    for t in CUTS:
        if score_identities(fcluster(L, t, 'distance'), p, c)['pair precision %'] >= SETTINGS['target_pair_precision']:
            best = t
    return best
print(len(VARIANTS), 'variants ready')


In [ ]:
# 3. Run the 50 tests for every model and variant. Each model is saved as it finishes.
for m, d in DATA.items():
    fn = f'{OUT}/tests_{m}.csv'
    if os.path.exists(fn):
        print(m, 'already done'); continue
    pid, cam = d['pid'], d['cam']
    X = {k: unit(d[k].astype(np.float32)) for k in ('all', 'oaf')}
    people = np.unique(pid); rng = np.random.default_rng(SETTINGS['seed']); rows = []
    for s in tqdm(range(SETTINGS['n_splits']), desc=m):
        perm = rng.permutation(people); half = np.isin(pid, perm[:len(people) // 2])
        G = [np.flatnonzero(half), np.flatnonzero(~half)]
        for v, (src, f) in VARIANTS.items():
            Ls = []
            for g in G:
                S = np.clip(X[src][g] @ X[src][g].T, -1, 1)
                Dm = np.clip(1 - S, 0, None) if f is plain else to_dist(f(S, cam[g]))
                Dm = (Dm + Dm.T) / 2; np.fill_diagonal(Dm, 0); Ls.append(tree(Dm))
            for c_, t_ in [(0, 1), (1, 0)]:
                r = score_identities(fcluster(Ls[t_], calib(Ls[c_], pid[G[c_]], cam[G[c_]]), 'distance'),
                                     pid[G[t_]], cam[G[t_]])
                r.update(model=m, variant=v, split=s, direction=c_); rows.append(r)
    pd.DataFrame(rows).to_csv(fn, index=False)
    print(m, 'saved')


In [ ]:
# 4. STANDALONE summary (reads Drive only): means over 50 tests, and paired tests
import glob, shutil
from scipy.stats import wilcoxon
R = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{OUT}/tests_*.csv'))])
order = ['all frames', 'camera z-norm', 'camera S-norm', 're-ranking',
         'OAF', 'OAF + camera z-norm', 'OAF + camera S-norm', 'OAF + re-ranking']
cols = ['IDF1 %', 'IDs per person', 'cross-camera recall %', 'pair precision %']
SUM = R.groupby(['model', 'variant'])[cols].mean().round(2).reindex(order, level=1)
print('MEAN OVER 50 TESTS (pair precision = merge accuracy actually reached; target 90%)')
display(SUM); SUM.to_csv(f'{OUT}/summary.csv')
pairs = [('OAF', 'all frames'), ('OAF', 'camera z-norm'), ('OAF', 'camera S-norm'), ('OAF', 're-ranking'),
         ('OAF + re-ranking', 're-ranking'), ('OAF + re-ranking', 'OAF'),
         ('OAF + camera z-norm', 'camera z-norm'), ('OAF + camera z-norm', 'OAF')]
out = []
for m in R.model.unique():
    P = R[R.model == m].pivot_table(index=['split', 'direction'], columns='variant', values='IDF1 %')
    for x, y in pairs:
        dlt = P[x] - P[y]
        out.append({'model': m, 'comparison': f'{x}  vs  {y}', 'mean IDF1 change': round(dlt.mean(), 2),
                    'better in % of tests': round(100 * (dlt > 0).mean()),
                    'p (Wilcoxon)': f'{wilcoxon(dlt).pvalue:.1e}' if (dlt != 0).any() else 'n/a'})
PT = pd.DataFrame(out); display(PT); PT.to_csv(f'{OUT}/paired.csv', index=False)
shutil.make_archive('/content/Alternatives_results', 'zip', OUT)
print('Saved to', OUT, '| zip: /content/Alternatives_results.zip  (please upload the zip here)')
